# Credit Card Customer Default Analysis

This notebook explores customer balances, payment behavior, and repayment status. It compares a linear probability regression with logistic regression and generates default probabilities for the test customers.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss, log_loss, mean_absolute_error, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Load the datasets

Update `DATA_DIR` if the CSV files are stored elsewhere.

In [4]:
DATA_DIR = Path.cwd() / 'inter-uni-datathon-stream-1-credit-card-clients'
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Missing training values: {train.isna().sum().sum()}')
print(f'Default rate: {train.default.mean():.2%}')
train.head()

Train shape: (24000, 25)
Test shape:  (6000, 24)
Missing training values: 0
Default rate: 22.12%


,client_id,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,CC_0000F52C3717,50000,2,1,2,26,1,2,0,0,...,39475,40187,40992,2200,1962,1562,1500,1474,2300,0
1,CC_0008C4E44BB9,150000,2,1,2,26,-1,-1,-1,0,...,18400,1527,1527,0,18600,0,1527,0,0,0
2,CC_000B4B1BEDF6,450000,2,2,1,38,-2,-2,-2,-2,...,17255,17515,0,15008,1200,5255,4566,0,3648,0
3,CC_000D492CE7AD,230000,2,2,2,30,0,0,0,0,...,108826,100862,92481,6000,6000,4000,4000,3500,4000,0
4,CC_000DCFA992C4,160000,2,2,2,34,-1,-1,-1,-1,...,13780,12297,12752,24000,8000,13780,12300,12752,6000,0


## 2. Engineer interpretable customer metrics

The engineered fields describe average balances, total payments, payment-to-balance behavior, credit utilization, and delinquency history.

In [ ]:
def add_customer_metrics(data):
    result = data.copy()
    bill_columns = [f'BILL_AMT{month}' for month in range(1, 7)]
    payment_columns = [f'PAY_AMT{month}' for month in range(1, 7)]
    status_columns = ['PAY_0'] + [f'PAY_{month}' for month in range(2, 7)]
    result['average_bill_balance'] = result[bill_columns].mean(axis=1)
    result['average_payment'] = result[payment_columns].mean(axis=1)
    result['total_bill_balance'] = result[bill_columns].sum(axis=1)
    result['total_payment'] = result[payment_columns].sum(axis=1)
    result['payment_to_bill_ratio'] = result['total_payment'] / (result['total_bill_balance'].abs() + 1)
    result['credit_utilization'] = result['average_bill_balance'] / (result['LIMIT_BAL'] + 1)
    result['maximum_delinquency'] = result[status_columns].max(axis=1)
    result['delinquent_months'] = (result[status_columns] > 0).sum(axis=1)
    return result

train = add_customer_metrics(train)
test = add_customer_metrics(test)
status_summary = train.groupby('default')[['average_bill_balance', 'average_payment', 'credit_utilization', 'delinquent_months']].mean().round(2)
status_summary

## 3. Prepare features and validation data

Categorical codes are one-hot encoded, while numeric features are standardized inside the scikit-learn pipeline. The split is stratified so the validation default rate matches the training distribution.

In [ ]:
target = 'default'
id_column = 'client_id'
categorical_columns = ['SEX', 'EDUCATION', 'MARRIAGE'] + [f'PAY_{period}' for period in [0, 2, 3, 4, 5, 6]]
feature_columns = [column for column in train.columns if column not in {target, id_column}]
x = train[feature_columns]
y = train[target]
x_train, x_valid, y_train, y_valid = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

def make_pipeline(model):
    categorical = [column for column in categorical_columns if column in feature_columns]
    numeric = [column for column in feature_columns if column not in categorical]
    transformer = ColumnTransformer([
        ('numeric', StandardScaler(), numeric),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical),
    ])
    return Pipeline([('preprocess', transformer), ('model', model)])

## 4. Compare regression models

The linear probability model directly answers the requested linear-regression use case. Logistic regression is included as a classification comparison. For probability prediction, ROC-AUC, log loss, Brier score, and mean absolute error are more informative than accuracy alone.

In [ ]:
def evaluate_model(name, model):
    model.fit(x_train, y_train)
    probabilities = np.clip(model.predict(x_valid), 0, 1)
    predictions = (probabilities >= 0.5).astype(int)
    print(name)
    print(f'  ROC-AUC:           {roc_auc_score(y_valid, probabilities):.4f}')
    print(f'  Average precision: {average_precision_score(y_valid, probabilities):.4f}')
    print(f'  Log loss:          {log_loss(y_valid, probabilities, labels=[0, 1]):.4f}')
    print(f'  Brier score:       {brier_score_loss(y_valid, probabilities):.4f}')
    print(f'  Accuracy:          {accuracy_score(y_valid, predictions):.4f}')
    print(f'  Mean absolute error: {mean_absolute_error(y_valid, probabilities):.4f}')
    return model

linear_model = evaluate_model('Linear probability regression', make_pipeline(LinearRegression()))
logistic_model = evaluate_model('Logistic regression comparison', make_pipeline(LogisticRegression(max_iter=2000)))

## 5. Fit the selected model and create the submission

The linear model is selected based on the supplied holdout results. Predictions are clipped to `[0, 1]` so they are valid default probabilities.

In [ ]:
linear_model.fit(x, y)
test_probabilities = np.clip(linear_model.predict(test[feature_columns]), 0, 1)
submission = pd.DataFrame({
    'client_id': test['client_id'],
    'default_probability': test_probabilities,
})
submission.to_csv('submission.csv', index=False)
print(f'Wrote {len(submission):,} predictions to submission.csv')
print(f'Probability range: {submission.default_probability.min():.4f} to {submission.default_probability.max():.4f}')
submission.head()